# InsertionSort (C++)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap12_tempo_esecuzione/InsertionSort_cpp.ipynb)

Capitolo 12 — Tempo di esecuzione. Notebook eseguibile di `InsertionSort`: l'algoritmo in C++, eseguito sugli esempi del capitolo, con la cella «Prova tu» per provarlo con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/analisi/12-tempo-esecuzione/).

Lo stesso notebook in [Python](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap12_tempo_esecuzione/InsertionSort_python.ipynb) e [Java](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap12_tempo_esecuzione/InsertionSort_java.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap12_tempo_esecuzione_cpp.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. Ogni programma C++ è scritto su un file dalla cella `%%writefile`, poi compilato con `g++` ed eseguito nella cella successiva: tutto avviene in Colab.

**Indici.** Nello pseudocodice gli array sono indicizzati da 1; nel codice C++ da 0: l'elemento `a[i]` dello pseudocodice è `a[i-1]` nel codice (e un ciclo `while i <= n` da 1 diventa `while i < n` da 0). Le tabelle stampate usano gli indici dello pseudocodice.

## Ordinamento per inserimento: InsertionSort

**Input:** un array $a[1..n]$ di $n$ numeri naturali. **Output:** l'array $a$ ordinato in modo non decrescente. Se $t(i)$ è il numero di spostamenti all'iterazione $i$, allora $S(n) = 6n - 4 + 3\sum_{i=2}^{n} t(i)$; nel caso peggiore (array decrescente) $S(n) = \frac{3n^2+9n-8}{2} = \Theta(n^2)$.

Pseudocodice: sezione 12.3 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/analisi/12-tempo-esecuzione/).

In [1]:
%%writefile insertion_sort.cpp
#include <cstdio>
#include <cmath>
#include <initializer_list>
#include <string>
using namespace std;

long long cnt[20];   // cnt[r] = numero di esecuzioni della riga r (numerazione delle dispense)

void Reset()
{
    for (int r = 0; r < 20; r++) cnt[r] = 0;
}

long long Total()
{
    long long s = 0;
    for (int r = 0; r < 20; r++) s += cnt[r];
    return s;
}

void PrintSteps(initializer_list<int> lines)
{
    printf("Riga | Esecuzioni\n");
    for (int r : lines) printf("%4d | %10lld\n", r, cnt[r]);
    printf("Passi totali S = %lld\n", Total());
}

// "[a[from], ..., a[to-1]]"
string Fmt(const int* a, int from, int to)
{
    string s = "[";
    for (int i = from; i < to; i++)
    {
        if (i > from) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

// InsertionSort(a, n)
void InsertionSort(int a[], int n, bool trace = false)
{
    long long prev = 0;                    // solo per la traccia
    int i = 1; cnt[2]++;                   // i := 2 nello pseudocodice
    cnt[3]++;
    while (i < n)                          // i <= n nello pseudocodice
    {
        int key = a[i]; cnt[4]++;
        int j = i - 1; cnt[5]++;
        cnt[6]++;                          // la condizione del while si conta a ogni valutazione
        while (j >= 0 && a[j] > key)       // j >= 1 and a[j] > key nello pseudocodice
        {
            a[j + 1] = a[j]; cnt[7]++;
            j = j - 1; cnt[8]++;
            cnt[6]++;
        }
        a[j + 1] = key; cnt[10]++;
        i = i + 1; cnt[11]++;
        cnt[3]++;
        if (trace)                         // traccia: dopo l'incremento, i e' l'indice dello pseudocodice dell'iterazione appena finita
        {
            printf("%2d | %3d | %4lld | %s\n", i, key, cnt[7] - prev, Fmt(a, 0, n).c_str());
            prev = cnt[7];
        }
    }
}

long long StepsOf(const int a[], int n)    // lavora su una copia di a
{
    int b[64];
    for (int i = 0; i < n; i++) b[i] = a[i];
    Reset();
    InsertionSort(b, n);
    return Total();
}

#ifndef SOLO_LIBRERIA
int main()
{
    // Esempio delle dispense: a[1..6] = (5, 2, 4, 6, 1, 3)
    int a[] = {5, 2, 4, 6, 1, 3};
    int n = 6;
    printf("a = %s\n", Fmt(a, 0, n).c_str());
    Reset();
    printf(" i | key | t(i) | array dopo l'iterazione\n");
    InsertionSort(a, n, true);
    printf("ordinato: %s\n", Fmt(a, 0, n).c_str());
    PrintSteps({2, 3, 4, 5, 6, 7, 8, 10, 11});
    long long t = cnt[7];
    printf("sum t(i) = %lld, S(n) = 6n - 4 + 3 sum t(i) = %lld\n", t, 6 * n - 4 + 3 * t);

    // Caso peggiore: array decrescente, t(i) = i - 1
    printf("\n");
    int d[] = {6, 5, 4, 3, 2, 1};
    printf("a = %s (decrescente): S = %lld = (3n^2 + 9n - 8)/2 = %d\n", Fmt(d, 0, n).c_str(), StepsOf(d, n), (3 * n * n + 9 * n - 8) / 2);
    printf("sum t(i) = %lld = n(n-1)/2 = %d\n", cnt[7], n * (n - 1) / 2);

    printf("\n");
    printf("Array decrescente (n, ..., 2, 1):\n");
    printf("  n | S(n) | (3n^2 + 9n - 8)/2\n");
    for (int k = 1; k <= 8; k++)
    {
        int b[8];
        for (int i = 0; i < k; i++) b[i] = k - i;
        printf("%3d | %4lld | %d\n", k, StepsOf(b, k), (3 * k * k + 9 * k - 8) / 2);
    }
    return 0;
}
#endif

Writing insertion_sort.cpp


In [2]:
!g++ -std=c++17 -O2 -o insertion_sort insertion_sort.cpp && ./insertion_sort

a = [5, 2, 4, 6, 1, 3]
 i | key | t(i) | array dopo l'iterazione
 2 |   2 |    1 | [2, 5, 4, 6, 1, 3]
 3 |   4 |    1 | [2, 4, 5, 6, 1, 3]
 4 |   6 |    0 | [2, 4, 5, 6, 1, 3]
 5 |   1 |    4 | [1, 2, 4, 5, 6, 3]
 6 |   3 |    3 | [1, 2, 3, 4, 5, 6]
ordinato: [1, 2, 3, 4, 5, 6]
Riga | Esecuzioni
   2 |          1
   3 |          6
   4 |          5
   5 |          5
   6 |         14
   7 |          9
   8 |          9
  10 |          5
  11 |          5
Passi totali S = 59
sum t(i) = 9, S(n) = 6n - 4 + 3 sum t(i) = 59

a = [6, 5, 4, 3, 2, 1] (decrescente): S = 77 = (3n^2 + 9n - 8)/2 = 77
sum t(i) = 15 = n(n-1)/2 = 15

Array decrescente (n, ..., 2, 1):
  n | S(n) | (3n^2 + 9n - 8)/2
  1 |    2 | 2
  2 |   11 | 11
  3 |   23 | 23
  4 |   38 | 38
  5 |   56 | 56
  6 |   77 | 77
  7 |  101 | 101
  8 |  128 | 128


**Prova tu.** Cambia l'array e controlla la formula $6n - 4 + 3\sum t(i)$.

In [3]:
%%writefile prova_insertion_sort.cpp
#define SOLO_LIBRERIA   // non eseguire il main di insertion_sort.cpp
#include "insertion_sort.cpp"

int main()
{
    int a[] = {8, 3, 5, 1, 9, 2, 7};      // <-- cambia l'array
    int n = 7;
    Reset();
    printf(" i | key | t(i) | array dopo l'iterazione\n");
    InsertionSort(a, n, true);
    PrintSteps({2, 3, 4, 5, 6, 7, 8, 10, 11});
    printf("6n - 4 + 3 sum t(i) = %lld\n", 6 * n - 4 + 3 * cnt[7]);
    return 0;
}

Writing prova_insertion_sort.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_insertion_sort prova_insertion_sort.cpp && ./prova_insertion_sort

 i | key | t(i) | array dopo l'iterazione
 2 |   3 |    1 | [3, 8, 5, 1, 9, 2, 7]
 3 |   5 |    1 | [3, 5, 8, 1, 9, 2, 7]
 4 |   1 |    3 | [1, 3, 5, 8, 9, 2, 7]
 5 |   9 |    0 | [1, 3, 5, 8, 9, 2, 7]
 6 |   2 |    4 | [1, 2, 3, 5, 8, 9, 7]
 7 |   7 |    2 | [1, 2, 3, 5, 7, 8, 9]
Riga | Esecuzioni
   2 |          1
   3 |          7
   4 |          6
   5 |          6
   6 |         17
   7 |         11
   8 |         11
  10 |          6
  11 |          6
Passi totali S = 71
6n - 4 + 3 sum t(i) = 71


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).